# Guardião da Família — notebook do Colab

O Guardião avisa um familiar responsável quando um **número novo** diz ser parente de uma
pessoa idosa e pede dinheiro (o golpe do falso parente). Este notebook instala o backend,
lê as credenciais dos **Secrets do Colab** e roda o Guardião em um de dois modos:

- **demonstração** (padrão): a conta da pessoa idosa é simulada, com contatos, conversas
  antigas e números fictícios (DDD 00, que não existe no Brasil). O Gemini, a busca nos
  contatos e o bot do Telegram que alerta a responsável continuam reais.
- **real** (opcional): lê a conta **de teste** do Telegram da pessoa idosa pelo Telethon.

O notebook segue as quatro fases do Guardião e os três testes do enunciado, nesta ordem:

| Seção | O que acontece |
|---|---|
| 1–5 | obter o código, instalar, ler os Secrets, escolher o modo e conectar o Guardião |
| 6 | a tela do familiar sobe, com o QR do bot para a responsável |
| 7 | a célula espera a responsável tocar **Iniciar** no bot |
| 8 | Filtra: o vizinho Carlos e o grupo são descartados, sem texto e sem caso |
| 9 | Interpreta: o Gemini lê o golpe clássico (Pedro, neto, Pix) e marca a suspeita |
| 10 | Confere: o alerta leva o número salvo do Pedro |
| 11 | Avisa: toque **É golpe** → caso golpe e aviso no celular da Dona Maria |
| 12 | teste 1: confere a interpretação, o alerta no bot, o toque e o sinal |
| 13 | teste 2: a Júlia diz ser neta e não pede dinheiro; o agente pede esse dado e não consulta o bot |
| 14 | teste 3: o bot falha uma vez no cenário do Lucas; nenhum caso; o envio volta |
| 15 | encerrar: apaga os casos e a sessão do Telethon |

**Executar tudo** funciona: as células de espera aguardam o Iniciar e o toque, com
tempo-limite e aviso. Como o Encerrar é a última célula, para manter a tela no ar no fim de
uma apresentação use *Ambiente de execução → Executar anteriores* na célula do Encerrar.

**Segurança:** nenhuma credencial é escrita neste notebook. Elas ficam nos Secrets do Colab
(ícone de chave na barra lateral) e o notebook só mostra se cada uma está *presente* ou
*ausente*. Nunca cole um token ou chave numa célula.

## 1. Obter o código

A célula abaixo procura a pasta do projeto (com `backend/` e `frontend/`) nesta ordem:

1. **pasta existente** — a pasta atual, as pastas acima dela ou uma pasta em `/content`;
2. **ZIP enviado** — um `.zip` do projeto enviado para `/content` (painel *Arquivos*), que é extraído;
3. **`git clone`** do repositório público, só no Colab. Não precisa de token.

In [ ]:
import os
import subprocess
import sys
import zipfile
from pathlib import Path

try:
    import google.colab  # noqa: F401

    EM_COLAB = True
except ImportError:
    EM_COLAB = False

REPOSITORIO = "https://github.com/fiap-poa/guardiao_gs_fiap.git"
CONTENT = Path("/content")
DESTINO_CLONE = CONTENT / "guardiao_gs_fiap"


def _eh_raiz(pasta):
    return (pasta / "backend" / "guardiao").is_dir() and (pasta / "frontend").is_dir()


def _procurar_raiz():
    candidatos = [Path.cwd(), *Path.cwd().parents]
    if CONTENT.is_dir():
        candidatos += sorted(p for p in CONTENT.glob("*") if p.is_dir())
        candidatos += sorted(p for p in CONTENT.glob("*/*") if p.is_dir())
    return next((p.resolve() for p in candidatos if _eh_raiz(p)), None)


def _extrair_zip():
    if not CONTENT.is_dir():
        return False
    for arquivo in sorted(CONTENT.glob("*.zip")):
        with zipfile.ZipFile(arquivo) as zip_:
            if any(nome.endswith("backend/guardiao/__init__.py") for nome in zip_.namelist()):
                destino = CONTENT / arquivo.stem
                zip_.extractall(destino)
                print(f"ZIP extraído: {arquivo.name} -> {destino}")
                return True
    return False


def _clonar():
    # repositório público: sem token; GIT_TERMINAL_PROMPT=0 faz o git falhar em vez de travar pedindo usuário
    ambiente = {**os.environ, "GIT_TERMINAL_PROMPT": "0"}
    comando = ["git", "clone", "--depth", "1", "-q", REPOSITORIO, str(DESTINO_CLONE)]
    print("clonando o repositório...")
    try:
        resultado = subprocess.run(comando, capture_output=True, text=True, env=ambiente, timeout=120)
        codigo, erro = resultado.returncode, resultado.stderr.strip()
    except subprocess.TimeoutExpired:
        codigo, erro = None, "tempo esgotado (120 s)"
    if codigo != 0:
        raise RuntimeError(f"git clone falhou (código {codigo}): {erro}")


RAIZ = _procurar_raiz()
if RAIZ is None and _extrair_zip():
    RAIZ = _procurar_raiz()
if RAIZ is None:
    if EM_COLAB:
        _clonar()
        RAIZ = _procurar_raiz()
    else:
        print("aviso: fora do Colab o clone é pulado; abra este notebook de dentro do repositório")
if RAIZ is None:
    raise RuntimeError("não achei a pasta do projeto (com backend/ e frontend/)")

BACKEND = RAIZ / "backend"
if str(BACKEND) not in sys.path:
    sys.path.insert(0, str(BACKEND))
print(f"projeto em {RAIZ} (Colab: {'sim' if EM_COLAB else 'não'})")

## 2. Instalar as dependências

No Colab, instala `backend/requirements.txt` (Flask, Telethon, cliente HTTP da Bot API…).
Fora do Colab a instalação é pulada: use o ambiente virtual do backend
(`backend/.venv`) como kernel.

In [ ]:
import importlib.util

REQUISITOS = BACKEND / "requirements.txt"
if EM_COLAB:
    instalacao = subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", str(REQUISITOS)], capture_output=True, text=True
    )
    if instalacao.returncode != 0:
        print("\n".join(instalacao.stderr.strip().splitlines()[-20:]))
        raise RuntimeError(f"pip install falhou (código {instalacao.returncode})")
    print("dependências instaladas")
else:
    faltando = [m for m in ("flask", "telethon", "requests", "dotenv", "qrcode") if importlib.util.find_spec(m) is None]
    try:
        tem_genai = importlib.util.find_spec("google.genai") is not None
    except ModuleNotFoundError:  # nem o pacote `google` existe
        tem_genai = False
    if not tem_genai:
        faltando.append("google-genai")
    if faltando:
        print(f"aviso: fora do Colab a instalação é pulada; faltam {faltando}: pip install -r {REQUISITOS}")
    else:
        print("aviso: fora do Colab a instalação é pulada (as dependências já estão no kernel)")

## 3. Credenciais (Secrets do Colab)

No Colab, abra o ícone de **chave** na barra lateral, crie os Secrets abaixo e ative
*Acesso ao notebook* em cada um:

| Secret | Para quê | Modo |
|---|---|---|
| `TELEGRAM_BOT_TOKEN` | bot do Guardião (alerta a responsável) | demonstração e real |
| `GEMINI_API_KEY` | detecção do falso parente nas mensagens de contato novo | demonstração e real |
| `GEMINI_MODELO` | opcional: outro modelo do Gemini (padrão `gemini-3.5-flash-lite`) | demonstração e real |
| `TELEGRAM_API_ID`, `TELEGRAM_API_HASH` | ler a conta de teste pelo Telethon | só real |

A célula copia os Secrets para variáveis de ambiente do processo e mostra só
**presente/ausente**. Fora do Colab, as credenciais vêm de `backend/.env` (fora do git).
Sem `TELEGRAM_BOT_TOKEN`, o modo demonstração avisa e segue sem bot: os cenários rodam,
mas nenhum alerta chega à responsável. Sem `GEMINI_API_KEY`, a detecção fica desligada:
o filtro funciona, mas nada é analisado.

In [ ]:
from guardiao.config import carregar_env, carregar_segredos_colab

SEGREDOS = ["TELEGRAM_BOT_TOKEN", "GEMINI_API_KEY", "GEMINI_MODELO", "TELEGRAM_API_ID", "TELEGRAM_API_HASH"]
do_colab = carregar_segredos_colab(SEGREDOS)
if not EM_COLAB:
    print("aviso: fora do Colab não há Secrets; lendo backend/.env\n")
    carregar_env()
for nome in SEGREDOS:
    presente = do_colab[nome] or bool(os.environ.get(nome, "").strip())
    print(f"{nome:20} {'presente' if presente else 'ausente'}")
if not os.environ.get("TELEGRAM_BOT_TOKEN", "").strip():
    print("\naviso: sem TELEGRAM_BOT_TOKEN o modo demonstração segue sem bot (nenhum alerta sai)")
if not os.environ.get("GEMINI_API_KEY", "").strip():
    print("\naviso: sem GEMINI_API_KEY a detecção pelo Gemini fica desligada")

## 4. Escolher o modo

- `"demonstracao"`: canal simulado. É o modo da entrega e não precisa de conta do Telegram.
- `"real"`: Telethon numa conta **de teste** do Telegram (nunca a conta de uma pessoa real).
  Precisa de `TELEGRAM_API_ID` e `TELEGRAM_API_HASH`; sem sessão salva, a conta é conectada
  pelo QR da tela (seção 6). É opcional.

In [ ]:
MODO = "demonstracao"  # ou "real"

from guardiao.canal.fabrica import MODOS

if MODO not in MODOS:
    raise ValueError(f"MODO deve ser um de {MODOS}")
print(f"modo: {MODO}")

## 5. Conectar o Guardião

O canal é a única parte que fala com o mensageiro; o **Guardião** recebe tudo o que ele
entrega. Ao conectar, ele:

1. registra como **conhecidos** os contatos que já tinham conversa privada com a pessoa
   idosa (e o próprio bot);
2. liga a recepção das mensagens;
3. manda o `/start` automático pela conta da pessoa idosa, para o bot poder avisá-la
   depois (o bot não responde a ela).

Daí em diante, só as mensagens de **contato novo** seguem para a análise (guardadas em
`ANALISADAS`). Grupos e conhecidos são descartados sem ler o texto. Com `GEMINI_API_KEY`,
cada mensagem de contato novo vai ao **Gemini** junto com as anteriores do mesmo número
nos últimos 30 minutos (só os textos: nenhum contato, telefone ou nome da agenda). Se a
pessoa diz ser parente **e** pede dinheiro, o Guardião busca nos contatos o número
verdadeiro desse parente e guarda a suspeita em `SUSPEITAS`.

Com a responsável cadastrada (seção 7), o roteiro mostra as quatro fases do Guardião:
**Filtra** (seção 8), **Interpreta** (seção 9), **Confere** (seção 10) e **Avisa**
(seção 11). Cada suspeita vira um **caso**: o alerta com os botões vai pelo bot a ela, e
o toque decide o desfecho (seção 11). `PRAZO_RESPOSTA` é quanto o Guardião espera o toque
antes de avisar a pessoa idosa (10 minutos), e `COMO_CHAMA_IDOSA` é como o alerta se
refere a ela ("sua mãe"). Os casos ficam em `backend/.local/casos.json` (modo 600, fora
do git) só com o trecho, quem a pessoa disse ser, o estado, a resposta e os horários;
são apagados em 30 dias ou no fim da demonstração (seção 15).

**Privacidade:** os textos analisados vão para o Google (Gemini API). No nível gratuito, os
termos da Gemini API permitem que o Google use esse conteúdo para melhorar os produtos.
No modo demonstração as mensagens são fictícias; no modo real, use só contas de teste da
equipe, nunca a conta de uma pessoa real. Depois vêm os três testes do enunciado: o sinal
do que as fases já produziram (seção 12), a Júlia sem pedido de dinheiro (seção 13) e a
falha controlada do bot (seção 14). Rodar a célula de novo fecha o Guardião anterior
antes de criar outro; os casos que ainda aguardavam o toque voltam como "sem resposta",
sem novo aviso.

**Modo real sem sessão salva:** o Guardião não conecta aqui, mas fica registrado para a tela
(seção 6). Quando a tela abre, o login por QR começa sozinho: lido o QR no celular da pessoa
idosa, o Guardião conecta sem rodar esta célula de novo. Se a conta tiver verificação em duas
etapas, a tela não pede a senha: faça o login pelo terminal, na pasta `backend/`
(`python -m guardiao.verificar login`), e rode esta célula de novo.

In [ ]:
from datetime import timedelta

from guardiao.analise import Analisador
from guardiao.app import definir_guardiao
from guardiao.canal.fabrica import criar_canal
from guardiao.config import LOCAL_DIR
from guardiao.gemini import criar_detector
from guardiao.recepcao import Guardiao, RelogioAjustavel

if globals().get("guardiao") is not None:
    guardiao.fechar()
elif globals().get("canal") is not None:
    canal.fechar()
canal = guardiao = None  # se a criação falhar, não sobra referência ao canal fechado
definir_guardiao(None)  # a tela não mostra o Guardião fechado se o novo não conectar

PRAZO_RESPOSTA = timedelta(minutes=10)  # sem toque da responsável nesse prazo, a pessoa idosa é avisada
COMO_CHAMA_IDOSA = "sua mãe"  # como o alerta se refere à pessoa idosa
CASOS_ARQUIVO = LOCAL_DIR / "casos.json"  # só trecho, quem diz ser, estado, resposta e horários

RELOGIO = RelogioAjustavel()  # o prazo de 10 minutos do caso usa este relógio
ANALISADAS = []  # mensagens de contato novo que seguem para a análise
RESPOSTAS = []  # toques da responsável nos botões dos alertas
SUSPEITAS = []  # contatos novos que disseram ser parente e pediram dinheiro

detector = criar_detector()  # None sem GEMINI_API_KEY
canal = criar_canal(MODO)
analisador = None
if detector is not None:
    analisador = Analisador(detector, canal.listar_contatos, ao_suspeitar=SUSPEITAS.append, relogio=RELOGIO)
guardiao = Guardiao(
    canal, relogio=RELOGIO, ao_analisar=ANALISADAS.append, ao_responder=RESPOSTAS.append, analisador=analisador,
    prazo_resposta=PRAZO_RESPOSTA, casos_arquivo=CASOS_ARQUIVO, como_chama_idosa=COMO_CHAMA_IDOSA,
)
try:
    estado = guardiao.conectar()
except Exception as erro:
    if MODO == "real" and type(erro).__name__ in ("SemSessao", "SessaoNaoAutorizada"):
        definir_guardiao(guardiao)  # sem sessão: a tela (seção 6) mostra o QR da conta e conecta ao ler
        print("conta sem sessão salva: o Guardião fica registrado, e a tela (seção 6) mostra o QR da conta")
        print("(conta com verificação em duas etapas: login pelo terminal, python -m guardiao.verificar login)")
    else:
        guardiao.casos.fechar()
        guardiao = None
        print(f"aviso: o Guardião não conectou ({type(erro).__name__})")
else:
    definir_guardiao(guardiao)  # a tela (seção 6) passa a mostrar este Guardião
    print(f"conectado: {estado['conhecidos']} contatos conhecidos no pareamento")
    print(f"pessoa idosa habilitada no bot: {'sim' if estado['idosa_habilitada'] else 'aguardando o /start'}")
    if detector is None:
        print("detecção pelo Gemini: desligada (sem GEMINI_API_KEY)")
    else:
        print(f"detecção pelo Gemini: ligada (modelo {detector.cliente.modelo})")
    print(f"casos guardados de execuções anteriores: {len(guardiao.casos.listar())}")
print(canal)

## 6. Tela do familiar

Sobe o servidor Flask numa thread e mostra a tela da responsável dentro do notebook com
`serve_kernel_port_as_iframe`. Ela lê o Guardião conectado na seção 5:

- **configuração**, enquanto não termina: o QR da **conta da pessoa idosa** (só no modo real
  sem sessão: no celular dela, Telegram → Configurações → Dispositivos → Conectar
  dispositivo; no modo demonstração a conta é simulada) e o QR do **bot da responsável**, com
  o link de uso único e o botão **Abrir o bot**. É por esse QR que a responsável se cadastra
  (seção 7). O link só muda quando falta pouco para vencer;
- **casos**, depois do Iniciar: horário, quem a pessoa disse ser, trecho, estado (aguardando,
  golpe, era ele mesmo, sem resposta) e horário do aviso à pessoa idosa, atualizados a cada
  poucos segundos. A tela não responde casos: o toque é no alerta do Telegram;
- **celular do golpista**, só no modo demonstração: injeta os cenários pela tela. As seções
  8 e 9 injetam o Carlos, o grupo e o Pedro.

A tela nunca pede a senha de duas etapas da conta. Os pedidos usam caminhos relativos, o que
funciona no iframe do Colab. Fora do Colab, abra o endereço impresso no navegador.

In [ ]:
from guardiao.app import definir_guardiao, iniciar_em_segundo_plano

definir_guardiao(guardiao)  # a tela mostra o estado deste Guardião

try:
    servidor = iniciar_em_segundo_plano()
except OSError as erro:
    servidor = None
    print(f"aviso: não consegui subir o Flask ({type(erro).__name__}: porta em uso?)")
if servidor is not None:
    print(f"Flask no ar na porta {servidor.port}")
    if EM_COLAB:
        from google.colab import output

        output.serve_kernel_port_as_iframe(servidor.port, path="/", height=900)
    else:
        print(f"aviso: fora do Colab não há iframe; abra http://127.0.0.1:{servidor.port}/")

### Link do bot

O mesmo link que a tela mostra no QR (o código vigente, `link_cadastro_vigente`), para quem
preferir abrir no Telegram da responsável sem ler o QR. É de uso único e vale 30 minutos.
Sem `TELEGRAM_BOT_TOKEN` não há bot, e a célula só avisa.

In [ ]:
from guardiao.cadastro import JaCadastrada
from guardiao.recepcao import SemBot

if guardiao is None:
    print("aviso: o Guardião não está conectado; rode a célula da seção 5")
else:
    try:
        LINK_CADASTRO = guardiao.link_cadastro_vigente()  # o mesmo código do QR da tela
    except SemBot as erro:
        print(f"aviso: {erro}")
    except JaCadastrada:
        print("a responsável já está cadastrada: não há link novo")
    except Exception as erro:
        print(f"aviso: não consegui falar com o bot ({type(erro).__name__})")
    else:
        print("No Telegram da responsável, leia o QR da tela ou abra o link abaixo e toque em Iniciar")
        print("(uso único, vale 30 minutos):")
        print(LINK_CADASTRO)

## 7. Cadastro da responsável

O bot é privado: só cadastra quem toca **Iniciar** pelo link (ou QR) com código de uso único
da seção 6. A responsável recebe "Pronto, …" pelo bot. Um Iniciar sem código, com código já
usado, vencido ou inventado recebe "Este bot é privado…" e não cadastra ninguém. Depois do
cadastro, não há link novo.

A célula **espera** o Iniciar por até `ESPERA_CADASTRO_S` segundos. Se ninguém tocar, ela
avisa e as células seguintes pulam com aviso: toque Iniciar e rode esta célula de novo. Sem
`TELEGRAM_BOT_TOKEN` não há o que esperar, e a célula só avisa.

In [ ]:
from guardiao.roteiro import descrever_estado, esperar_cadastro

ESPERA_CADASTRO_S = 300  # quanto a célula espera o Iniciar da responsável

esperar_cadastro(guardiao, ESPERA_CADASTRO_S)
if guardiao is not None:
    print()
    print(descrever_estado(guardiao))

## 8. Filtra

A pessoa idosa simulada é a **Dona Maria**. O Guardião descarta, sem ler o texto, o que
não é conversa privada de um contato novo:

| Cenário | O que acontece | Por quê |
|---|---|---|
| `contato_antigo_pede_dinheiro` | o vizinho Carlos, que já conversava com ela, pede dinheiro | conhecido: descartado sem ler o texto |
| `mensagem_em_grupo` | alguém no grupo Família pede Pix | o Guardião não lê grupos |

Nenhum dos dois vira caso. Do que é descartado, o texto não é lido nem mostrado.

In [ ]:
from guardiao.canal.cenarios import obter_cenario
from guardiao.roteiro import falta_para_alertar, injetar_mostrando

CENARIOS_FILTRA = ("contato_antigo_pede_dinheiro", "mensagem_em_grupo")

falta = falta_para_alertar(guardiao, MODO)
if falta:
    print(f"aviso: {falta}")
else:
    for cenario_id in CENARIOS_FILTRA:
        cenario = obter_cenario(cenario_id)
        print(f"== {cenario_id}")
        injetar_mostrando(guardiao, ANALISADAS, canal.injetar, cenario_id)
        aberto = guardiao.casos.caso_de(cenario.remetente_id) is not None
        print(f"   alerta: {'SIM (não devia)' if aberto else 'nenhum'}")

## 9. Interpreta

O golpe clássico: um número novo diz ser o neto **Pedro** e pede um Pix. Como o número
é novo, a conversa segue para a análise. O **Gemini** recebe só os textos e diz se a
pessoa alega ser parente e se pede dinheiro. Com as duas coisas, há suspeita.

O Guardião não responde ao golpista, não pede dado nenhum a ele e não fala com o parente.

In [ ]:
from guardiao.roteiro import ESTADOS_CASO, abrir_caso, falta_para_alertar, mostrar_deteccao

falta = falta_para_alertar(guardiao, MODO)
if falta:
    print(f"aviso: {falta}")
else:
    print("== golpe_classico")
    ja_aberto = guardiao.casos.caso_de("sim:novo-1") is not None
    caso = abrir_caso(guardiao, ANALISADAS, "golpe_classico")
    if caso is None:
        print("aviso: o alerta não saiu (veja o aviso acima); rode a célula de novo")
    elif ja_aberto:
        print(f"   caso já aberto, dizendo ser {caso.quem}; nenhum alerta novo")
        print(f"   caso: {ESTADOS_CASO[caso.estado]}")
    else:
        print(f"   alerta enviado à responsável: número novo dizendo ser {caso.quem}")
        print(f"   caso: {ESTADOS_CASO[caso.estado]}")
    print()
    mostrar_deteccao(guardiao, SUSPEITAS)

## 10. Confere

A suspeita do Pedro já foi marcada (seção 9). A busca nos contatos da Dona Maria acha o
número salvo desse nome, e o alerta leva esse número:

`+55 00 98765-0003 (salvo como Pedro Neto)`

Nada é pedido ao golpista. Sem o número salvo, o alerta seguiria sem número.

In [ ]:
from guardiao.roteiro import falta_para_alertar, numero_esperado_pedro, numero_no_alerta

falta = falta_para_alertar(guardiao, MODO)
if falta:
    print(f"aviso: {falta}")
else:
    esperado = numero_esperado_pedro(canal)
    no_alerta = numero_no_alerta(SUSPEITAS, "sim:novo-1")
    caso = guardiao.casos.caso_de("sim:novo-1")
    print(f"número salvo: {esperado or 'nenhum'}")
    print(f"número verdadeiro no alerta: {no_alerta or 'nenhum'}")
    if caso is None:
        print("aviso: a suspeita do Pedro ainda não gerou alerta; rode a seção 9")
    elif no_alerta is None:
        print("aviso: o alerta seguiu sem o número salvo")
    elif no_alerta == esperado:
        print("a conferência achou o número salvo do Pedro; nada foi pedido ao golpista")
    else:
        print("aviso: o número do alerta não é o salvo do Pedro")

## 11. Avisa

O alerta do Pedro está no Telegram da responsável e o caso está **aguardando** o toque.
Toque **É golpe**: o caso fica **golpe** e o aviso aparece no celular simulado da Dona
Maria.

A célula espera o toque de verdade por até `ESPERA_TOQUE_S` segundos. Com
`TOQUE_SIMULADO = True`, ela simula o toque pelo mesmo caminho do bot, sem abrir o
Telegram. Sem toque no prazo, o caso continua **aguardando**: toque no alerta e rode a
célula de novo. O texto do que a Filtra descartou não é mostrado.

In [ ]:
from guardiao.canal.cenarios import obter_cenario
from guardiao.canal.simulado import formatar_celular
from guardiao.roteiro import ESTADOS_CASO, falta_para_alertar, tocar

TOQUE_SIMULADO = False  # True: simula o toque da responsável, sem abrir o Telegram
ESPERA_TOQUE_S = 180  # quanto a célula espera o toque de verdade

falta = falta_para_alertar(guardiao, MODO)
if falta:
    print(f"aviso: {falta}")
else:
    caso = guardiao.casos.caso_de(obter_cenario("golpe_classico").remetente_id)
    if caso is None:
        print("aviso: o alerta do Pedro não está aberto; rode a seção 9")
    else:
        if caso.estado == "aguardando":
            if not TOQUE_SIMULADO:
                ESPERANDO_TOQUE = True  # a célula passa a esperar o toque
            caso = tocar(guardiao, caso, "golpe", simulado=TOQUE_SIMULADO, espera_s=ESPERA_TOQUE_S)
        print(f"   caso: {ESTADOS_CASO[caso.estado]}")
        if caso.estado == "aguardando":
            print('   ainda sem toque: toque "É golpe" no Telegram e rode a célula de novo')
        avisos = [linha for linha in canal.celular() if linha.tipo == "aviso"]
        print()
        print(formatar_celular(avisos))

## 12. Teste 1

Não injeta o Pedro de novo. Lê o que as quatro fases já produziram: a interpretação do
Gemini, o alerta que chegou ao bot e o toque **É golpe** no aviso e no caso. No fim,
confere o sinal (bot, Gemini, responsável, alerta com o número do Pedro, caso golpe,
aviso, tela e os descartes da Filtra). **OK**, ou a lista do que falhou.

In [ ]:
from guardiao.canal.cenarios import obter_cenario
from guardiao.roteiro import conferir_sinal, numero_no_alerta

remetente = obter_cenario("golpe_classico").remetente_id
analisador = guardiao.analisador if guardiao is not None else None
resultados = [r for r in analisador.ultimos() if str(r.remetente_id) == remetente] if analisador else []
analise = resultados[-1].analise if resultados and resultados[-1].analise is not None else None
if (
    analise is not None
    and analise.suspeita
    and analise.nome_alegado == "Pedro"
    and analise.parentesco == "neto"
    and analise.pede_dinheiro
    and analise.forma_pagamento == "pix"
):
    print("interpretação conferida: diz ser Pedro, neto, e pede Pix")
else:
    print("interpretação: não conferida")

canal_atual = globals().get("canal")
bot = getattr(canal_atual, "saida", None) if canal_atual is not None else None
if bot is not None and hasattr(bot, "alertas"):
    no_bot = any(getattr(alerta, "nome_alegado", None) == "Pedro" for alerta in bot.alertas)
elif (
    guardiao is not None
    and guardiao.casos.caso_de(remetente) is not None
    and numero_no_alerta(SUSPEITAS, remetente)
):
    no_bot = True
else:
    no_bot = False
print("consulta ao bot conferida: o alerta do Pedro chegou" if no_bot else "consulta ao bot: alerta do Pedro ausente")

caso = guardiao.casos.caso_de(remetente) if guardiao is not None else None
avisos = []
if canal_atual is not None and hasattr(canal_atual, "celular"):
    avisos = [linha for linha in canal_atual.celular() if linha.tipo == "aviso"]
toque_no_caso = caso is not None and caso.estado == "golpe" and caso.resposta == "golpe"
toque_no_aviso = caso is not None and caso.avisado_em is not None and any(caso.quem in linha.texto for linha in avisos)
print("toque conferido no aviso e no caso" if toque_no_caso and toque_no_aviso else "toque: não conferido no aviso e no caso")

SINAL_OK = conferir_sinal(guardiao, SUSPEITAS, globals().get("servidor"), MODO)

## 13. Teste 2

A Júlia escreve de um número novo, diz ser a neta e não pede dinheiro. O Gemini vê o
parentesco e a falta do pedido. O agente pede esse dado e para: sem ele não há suspeita,
nenhum caso e nenhuma consulta ao bot. O notebook não inventa o pedido nem um número.

In [ ]:
from guardiao.canal.cenarios import obter_cenario
from guardiao.roteiro import falta_para_alertar, injetar_mostrando, mostrar_deteccao

falta = falta_para_alertar(guardiao, MODO)
if falta:
    print(f"aviso: {falta}")
else:
    cenario = obter_cenario("contato_novo_sem_dinheiro")
    print(f"== {cenario.id}")
    injetar_mostrando(guardiao, ANALISADAS, canal.injetar, cenario.id)
    resultados = [r for r in guardiao.analisador.ultimos() if str(r.remetente_id) == str(cenario.remetente_id)]
    analise = resultados[-1].analise if resultados else None
    if analise is not None and analise.alega_parentesco and not analise.pede_dinheiro:
        print(
            f"diz ser {analise.nome_alegado}, {analise.parentesco}; falta o pedido de dinheiro. "
            "Qual foi o pedido: Pix, transferência ou pagamento de conta? "
            "Sem esse dado eu não consulto a responsável e não invento o pedido."
        )
    else:
        print("aviso: a leitura da Júlia não foi a esperada")
    suspeita = bool(resultados and resultados[-1].suspeita)
    print(f"suspeita: {'sim' if suspeita else 'não'}")
    aberto = guardiao.casos.caso_de(cenario.remetente_id) is not None
    print(f"alerta: {'SIM (não devia)' if aberto else 'nenhum'}")
    mostrar_deteccao(guardiao, SUSPEITAS)

## 14. Teste 3

Falha controlada da API do bot, não do Gemini. No cenário do Lucas, `enviar_alerta`
levanta uma vez: o erro é identificado, o caso não fica aberto e nada é inventado no
lugar. No `finally`, o envio volta ao método original.

In [ ]:
from guardiao.canal.cenarios import obter_cenario
from guardiao.roteiro import falta_para_alertar, injetar_mostrando

falta = falta_para_alertar(guardiao, MODO)
if falta:
    print(f"aviso: {falta}")
else:
    estado_envio = {"falhou": False, "original": canal.enviar_alerta}

    def falha_uma_vez(alerta):
        if not estado_envio["falhou"]:
            estado_envio["falhou"] = True
            raise ConnectionError
        return estado_envio["original"](alerta)

    canal.enviar_alerta = falha_uma_vez
    try:
        cenario = obter_cenario("parente_trocou_numero")
        print(f"== {cenario.id}")
        injetar_mostrando(guardiao, ANALISADAS, canal.injetar, cenario.id)
        if estado_envio["falhou"]:
            print("erro identificado: ConnectionError")
        else:
            print("aviso: o envio do alerta não falhou")
        aberto = guardiao.casos.caso_de(cenario.remetente_id) is not None
        print(f"caso do Lucas: {'aberto (não devia)' if aberto else 'nenhum'}")
    finally:
        canal.enviar_alerta = estado_envio["original"]
    print(f"envio restaurado: {canal.enviar_alerta is estado_envio['original']}")

## 15. Encerrar

Apaga os casos (da memória e de `backend/.local/casos.json`), fecha o Guardião (para o
polling do bot e desconecta o canal), desliga a tela e apaga a sessão do Telethon
(`backend/.local/guardiao.session*`): no modo real, faz o *log out* da conta no Telegram
antes de apagar. Roda mesmo sem Guardião conectado. Rode ao terminar a demonstração.

Num *Executar tudo*, esta célula roda no fim e derruba a tela. Para manter a tela no ar,
use *Ambiente de execução → Executar anteriores* nesta célula e rode-a depois.

In [ ]:
from guardiao.app import definir_guardiao, parar_servidor
from guardiao.casos import RepositorioCasos
from guardiao.config import LOCAL_DIR
from guardiao.roteiro import SESSAO, apagar_sessao

canal_encerrado = globals().get("canal")
if globals().get("guardiao") is not None:
    guardiao.casos.apagar_tudo()  # fim da demonstração: memória e arquivo
    guardiao.fechar()
elif canal_encerrado is not None:
    canal_encerrado.fechar()
canal = guardiao = None
definir_guardiao(None)
parar_servidor()
RepositorioCasos(globals().get("CASOS_ARQUIVO") or LOCAL_DIR / "casos.json").apagar()  # mesmo sem Guardião
sessao = apagar_sessao(canal_encerrado, LOCAL_DIR)  # modo real: log_out e apaga; as sobras saem pelo glob
print(f"Guardião encerrado; casos apagados; sessão do Telethon: {SESSAO[sessao]}")